# Step 0b — LITERATURE HARVEST · burst #21 `bn110920546` (GRB 110920A)
**Status: FINALIZED** — approved by VIKAS with feedback, then re-presented and approved.

Source of the step-0b results. Everything recomputes from the repo at run time.

In [1]:
import os, json, glob, hashlib, re, numpy as np
import astropy.io.fits as fits
from astropy.table import Table
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
BURST = "bn110920546"
sha = lambda p: hashlib.sha256(open(p,"rb").read()).hexdigest()
rel = lambda p: os.path.join(ROOT, p)
appr = json.load(open(rel(f"results/sweep106/{BURST}/APPROVALS.json")))
print("repo:", ROOT, "| burst:", BURST)

repo: /Users/salim/Desktop/Projects/SingleRest/Two_Breaks | burst: bn110920546


In [2]:
STEP = "0b"

In [3]:
s = appr[STEP]
print(f'step {STEP}: {s["status"]}  by {s["by"]}  {s["utc"]}')
for f in s.get("feedback", []):
    print(f'\n  PI feedback: {f["text"][:400]}')
    if f.get("routed"): print(f'  routed -> {f["routed"][:300]}')

step 0b: APPROVED  by VIKAS  2026-08-31T00:24:07Z

  PI feedback: find and add the missed paper carrying the published variability limit for this burst — it is load-bearing, because our old timing number conflicts with it, so that comparison must be in the dossier. Also list the four unreadable papers with the reason each failed. Then re-present 0b and I will approve.
  routed -> DONE 2026-08-30: dossier §4.1a (Golkhou+2015 limit + load-bearing comparison) + §7a (unreadable papers w/ reasons); harvest.json amendment_2026_08_30; LiteratureHarvest.md traps T10/T11 + VizieR recovery channel + checklist item; NR-32 published-limit screen already registered


## 1. What the harvest found

In [4]:
h = json.load(open(rel(f"notes/reconciliation/{BURST}_harvest.json")))
for k in ("grb_name","n_papers","n_papers_refereed","n_pdfs_new_this_pass","n_pdfs_not_obtained"):
    print(f'  {k:<24} {h.get(k)}')
print("\nZero-circular finding:", h["zero_circulars_finding"][:200])

  grb_name                 GRB 110920A
  n_papers                 35
  n_papers_refereed        31
  n_pdfs_new_this_pass     16
  n_pdfs_not_obtained      6

Zero-circular finding: n_circulars = 0. Verified by two independent endpoints with a positive control (110721A -> 9). Recorded as a RESULT per GCNIntelligence.md. Consequence: no GCN-sourced position, redshift, or afterglow


## 2. The paper that was MISSED, and why it was load-bearing

Golkhou+2015 carries a published variability **upper limit** for this trigger. It was missed
by every ADS full-text form because the paper's text names no individual triggers — its burst
membership lives only in the machine-readable table. Our shipped variability number violated
that limit.

In [5]:
a = h.get("amendment_2026_08_30", {})
print("added   :", a.get("paper_added", {}).get("bibcode"), "-", a.get("paper_added", {}).get("title"))
print("row     :", a.get("paper_added", {}).get("row"))
print("\nwhy missed:")
for c in a.get("missed_cause", []): print("  -", c)
print("\nconflict:", a.get("load_bearing_conflict"))

added   : 2015ApJ...811...93G - The Energy Dependence of GRB Minimum Variability Timescales
row     : 110920546 110920A dtmin 2.096 s f_dtmin=< (UPPER LIMIT); T90 160.771+-5.221 matches GBM catalog (identity by trigger number)

why missed:
  - machine-readable-table-only membership: the paper TEXT names no triggers, so all four ADS full-text forms miss it (the FLOOR warning class, now trap T10 with a VizieR recovery channel)
  - the PDF was ALREADY LOCAL under another burst tag (_bn120119170.pdf) — filename-based dedup cannot see cross-burst membership (trap T11: dedup by BIBCODE)

conflict: catalog MVT_S 5.342+-0.107 s (in-chain Haar, was labelled detection) VIOLATES the published upper limit <2.096 s by 2.5x; CWT 0.724 s consistent; canonical Bala pending; MVT_S is STALE-PENDING-REWALK (PI ruling 5) and NR-32 screens future rows


In [6]:
# the published row, read from the VizieR extraction in the family project
tsv = os.path.expanduser("~/Desktop/LATBright/GRB260226A/data/golkhou2015_table2.tsv")
for line in open(tsv):
    if line.startswith(("110920546","110920338")):
        f = line.rstrip("\n").split("\t")
        print(f"  trigger {f[0]}  name {f[1]}  dt_min {f[2]}  T90 {f[4]}  limit-flag {f[7] if len(f)>7 else ''}")
print("\nOurs is 110920546 — flagged '<', i.e. an UPPER LIMIT. The other row is a different burst\n"
      "sharing the catalog name (trap T1): match on trigger number, never on name.")

  trigger 110920338  name 110920A  dt_min  0.337  T90   9.728  limit-flag 
  trigger 110920546  name 110920A  dt_min  2.096  T90 160.771  limit-flag <

Ours is 110920546 — flagged '<', i.e. an UPPER LIMIT. The other row is a different burst
sharing the catalog name (trap T1): match on trigger number, never on name.


## 3. The four papers we could not read, with the reason each failed

In [7]:
for p in h["papers"]:
    if str(p.get("pdf") or "").strip(): continue
    src = str(p.get("pdf_source") or "")
    if "NOT OBTAINED" in src or "disappeared" in src:
        print(f'  {p["bibcode"]}  {p["first_author"]} {p["year"]}')
        print(f'      {src[:190]}\n')

  2024arXiv240115632C  Crupi, Riccardo 2024
      A file 'Crupi_2024_2024arXiv240115632C_bn101126198.pdf' was present in Skills_training at 17:48 UTC and had disappeared by 17:57 (only Crupi_2023_2023ExA56421C_bn101126198.pdf remains) - CON

  2025ApJ...983..130L  Liao, Tong-Lei 2025
      NOT OBTAINED: IOP served a Radware bot-manager captcha for doi 10.3847/1538-4357/adc00b; no arXiv version exists (arXiv title search returned 0).

  2024AN....34530179M  Ma, Guan-Lun 2024
      NOT OBTAINED: no arXiv version; Wiley/AN not open access; ADS link_gateway PUB_PDF failed.

  2017ApJS..229...31H  Hurley, K. 2017
      NOT OBTAINED: IOP bot-block on link_gateway PUB_PDF; no arXiv version listed by ADS. (I first guessed an arXiv ID, downloaded the WRONG paper - a cosmology preprint - and DELETED it; recorde



## 4. Frozen predictions (P0)

Frozen **before** any comparison, so a later match cannot be a retrofit.

In [8]:
p0 = json.load(open(rel(f"notes/reconciliation/{BURST}_P0_frozen.json")))
print("status:", p0["P0_STATUS"])
print("frozen:", str(p0["frozen_utc"])[:120])
print("predictions:", len(p0["predictions"]))

status: ARCHIVAL_POSTFIT
frozen: 2026-08-12 (doc-layer pass). Frozen BEFORE any sweep106 fit table was opened - this worker is HARD-BARRED from results/s
predictions: 9


## 5. Lessons this step produced
**T10** — a population paper's burst membership often lives only in its machine-readable
table, invisible to ADS full text; a VizieR membership sweep is now a Phase-1 recovery channel.
**T11** — the Golkhou PDF was already on disk under *another* burst's filename tag, defeating
filename dedup; dedup is now by bibcode. Both are in `dev/ai_guides/LiteratureHarvest.md`.